In [1]:
import os
import sys
import pandas as pd
import argparse
from datetime import datetime, timedelta
import pathlib

# 新增：將上一層目錄加入 sys.path
parent_path = pathlib.Path().absolute().parent
sys.path.insert(0, str(parent_path))
# Setup Django environment
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'website_configs.settings')
import django
django.setup()
# 重要：設定環境變數以允許在 Jupyter 的異步環境中執行同步操作
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

# Now we can import Django models
from app_user_keyword_db.models import NewsData
from app_top_person_db.models import TopPerson

In [2]:
from django.db.models import Q, Max, F
from collections import Counter

# Calculate top people from db and save them to db

In [9]:
latest_date = NewsData.objects.aggregate(max_date=Max('date'))['max_date']

# Calculate start date
start_date = latest_date - timedelta(weeks=12)  # 4 weeks ago

In [10]:
category = '即時'  # Example category (or "全部" for all categories)
entities_list = list(NewsData.objects.filter(category=category).filter(date__gte=start_date, date__lte=latest_date).values_list('entities', flat=True))

In [11]:
entities_list

["[NerToken(word='1', ner='CARDINAL', idx=(0, 1)), NerToken(word='70歲', ner='DATE', idx=(2, 5)), NerToken(word='陳', ner='PERSON', idx=(5, 6)), NerToken(word='花蓮', ner='GPE', idx=(45, 47)), NerToken(word='數10', ner='CARDINAL', idx=(59, 62)), NerToken(word='下午', ner='TIME', idx=(71, 73)), NerToken(word='5', ner='CARDINAL', idx=(78, 79)), NerToken(word='10日', ner='DATE', idx=(119, 122)), NerToken(word='5', ner='CARDINAL', idx=(123, 124)), NerToken(word='11日', ner='DATE', idx=(143, 146)), NerToken(word='上午', ner='TIME', idx=(146, 148)), NerToken(word='11.4公里', ner='QUANTITY', idx=(156, 162)), NerToken(word='當天下午', ner='TIME', idx=(198, 202)), NerToken(word='9天', ner='DATE', idx=(216, 218)), NerToken(word='今天', ner='DATE', idx=(224, 226)), NerToken(word=' 70歲', ner='PERSON', idx=(319, 323)), NerToken(word='4月11日', ner='DATE', idx=(327, 332)), NerToken(word='瓦拉米步道', ner='ORG', idx=(340, 345)), NerToken(word='今天', ner='DATE', idx=(348, 350)), NerToken(word='下午', ner='TIME', idx=(350, 352)), N

In [12]:

allowedNE=['PERSON']
news_categories=['即時', '要聞', '運動', '社會', '全球', '產經', '地方']
def ne_word_frequency( a_news_ne ):
    filtered_words =[]
    for ner,word in a_news_ne:
        if (len(word) >= 2) & (ner in allowedNE):
            filtered_words.append(word)
    counter = Counter( filtered_words )
    return counter.most_common( 20 )

# NerToken(word='烏克蘭', ner='GPE', idx=(4, 7))  # call function NerToken with three parameters: word, ner, and idx
def NerToken(word, ner, idx):
    # print(ner,word)
    return ner,word

In [14]:
    # Get the latest date in the database
latest_date = NewsData.objects.aggregate(max_date=Max('date'))['max_date']

# Calculate start date
start_date = latest_date - timedelta(weeks=12)  # 4 weeks ago


In [15]:

top_cate_ner_words={}
words_all=[]
for category in news_categories:
    entities_list=list(NewsData.objects.filter(category=category).filter(date__gte=start_date, date__lte=latest_date).values_list('entities', flat=True))
    words_group = []
    for entities in entities_list:
        if entities:
            words_group += eval(entities)

    words_all += words_group

    # Get top words by calling ne_word_frequency() function
    topwords = ne_word_frequency( words_group )
    top_cate_ner_words[category] = topwords


In [16]:

topwords_all = ne_word_frequency(words_all)
top_cate_ner_words['全部'] = topwords_all


In [17]:
top_cate_ner_words

{'即時': [('張祐銘', 6),
  ('唐斯', 6),
  ('許朝凱', 5),
  ('林鋅杰', 4),
  ('陳正興', 4),
  ('川普', 3),
  ('陳文杰', 3),
  ('李四川', 3),
  ('葉君璋', 2),
  (' 林鋅杰', 2),
  ('魏碩成', 2),
  ('林吳晉瑋', 2),
  ('劉基鴻', 2),
  ('林子偉', 2),
  ('顏郁軒', 2),
  ('張嘉玲', 2),
  ('徐堂琪', 2),
  ('尼克', 2),
  ('朱育賢', 2),
  (' 70歲', 1)],
 '要聞': [('李四川', 14),
  ('柯文哲', 11),
  ('許朝凱', 10),
  ('余登發', 7),
  ('李明璇', 7),
  ('楊智伃', 6),
  ('余陳月瑛', 5),
  ('陳冠廷', 5),
  ('謝國樑', 5),
  ('林國漳', 5),
  ('張嘉玲', 4),
  ('連勝文', 4),
  ('賴清德', 3),
  ('陳其邁', 3),
  ('趙少康', 3),
  ('王定宇', 3),
  ('王鴻薇', 3),
  ('洪子凱', 3),
  ('童子瑋', 3),
  ('張嘉郡', 3)],
 '運動': [('唐斯', 12),
  ('詹姆斯', 8),
  ('林鋅杰', 7),
  ('張祐銘', 6),
  ('李灝宇', 6),
  ('鈴木駿', 4),
  ('張宥謙', 4),
  ('葉子霆', 4),
  ('孔念恩', 4),
  ('葉君璋', 4),
  (' 林鋅杰', 4),
  ('陳正興', 4),
  ('徐堂琪', 4),
  ('尼克', 4),
  ('曾頌恩', 4),
  ('陳文杰', 3),
  ('史奈德', 3),
  ('肯納德', 3),
  ('林岳谷', 2),
  ('邦力多', 2)],
 '社會': [('阿華', 18),
  ('伍麗華', 8),
  ('許婦', 5),
  ('張男', 4),
  ('陳世煌', 3),
  (' 70歲', 2),
  ('簡男', 2),
  ('邱瑞杰', 2),
  ('王男', 2),
  ('林男

## Save or update top person table

In [19]:

# save it to db
# save it to db
for category, top_ners in top_cate_ner_words.items():
    # Convert the list of tuples to string representation for storage
    top_keys_str = str(top_ners)
    
    # Check if an entry for this category already exists
    try:
        # Update existing record
        obj = TopPerson.objects.get(category=category)
        obj.top_keys = top_keys_str
        obj.save()
    except TopPerson.DoesNotExist:
        # Create new record
        TopPerson.objects.create(
            category=category,
            top_keys=top_keys_str
        )

# Get category top people from db

In [21]:

import ast
# get charting data from database
def get_category_topPerson_db(cate, topk):

    # wf_pairs = data[cate][0:topk] # query from dataframe
    
    # SQL
    # statement = "SELECT top_keys FROM topperson where category='{}'".format(cate)
    # result = conn.execute(text(statement)).fetchone()
    # wf_pairs = eval(result[0])[0:topk]
    
    # ORM    
    queryset = TopPerson.objects.filter(category=cate).values('top_keys')
    if queryset.exists():
        top_keys_str = queryset[0]['top_keys']
        wf_pairs = ast.literal_eval(top_keys_str)[0:topk]
    else:
        wf_pairs = []    
    
    words = [w for w, f in wf_pairs]
    freqs = [f for w, f in wf_pairs]
    chart_data = {
        "category": cate,
        "labels": words,
        "values": freqs}
    #print(chart_data)
    return chart_data, wf_pairs  # chart_data

In [22]:
get_category_topPerson_db('即時', 5)

({'category': '即時',
  'labels': ['張祐銘', '唐斯', '許朝凱', '林鋅杰', '陳正興'],
  'values': [6, 6, 5, 4, 4]},
 [('張祐銘', 6), ('唐斯', 6), ('許朝凱', 5), ('林鋅杰', 4), ('陳正興', 4)])

# views.py

In [26]:
from django.shortcuts import render
import pandas as pd

from django.http import JsonResponse
from django.views.decorators.csrf import csrf_exempt
from django.contrib import messages
from django.shortcuts import redirect
from app_top_person_db.models import TopPerson
'''
the format of data:
{'政治': [('韓國瑜', 6344),
  ('蔡英文', 2114),
  ('賴清德', 1480),
  ...
  }
'''


def home(request):
    return render(request, 'app_top_person/home.html')
    # return render(request, 'app_top_person_db/home.html') # 注意: home.html是相同的，不要重複寫

# csrf_exempt is used for POST
# 單獨指定這一支程式忽略csrf驗證
@csrf_exempt
def api_get_topPerson(request):

    cate = request.POST.get('news_category')
    topk = request.POST.get('topk')
    topk = int(topk)

    chart_data, wf_pairs = get_category_topPerson_db(cate, topk)
    #chart_data, wf_pairs = get_category_topPerson(cate, topk)

    print("wf_pairs:",wf_pairs)
    
    if not wf_pairs:
        # If no data is found, return an error message
        response = {'error': 'No data found for the specified category.'}
        return JsonResponse(response)
    # If data is found, return the chart data and word-frequency pairs
    response = {'chart_data':  chart_data,
                'wf_pairs': wf_pairs,
                }
    return JsonResponse(response)


import ast
# get charting data from database
def get_category_topPerson_db(cate, topk):

    # wf_pairs = data[cate][0:topk] # query from dataframe
    
    # SQL
    # statement = "SELECT top_keys FROM topperson where category='{}'".format(cate)
    # result = conn.execute(text(statement)).fetchone()
    # wf_pairs = eval(result[0])[0:topk]
    
    # ORM    
    queryset = TopPerson.objects.filter(category=cate).values('top_keys')
    if queryset.exists():
        top_keys_str = queryset[0]['top_keys']
        wf_pairs = ast.literal_eval(top_keys_str)[0:topk]
    else:
        wf_pairs = []    
    
    words = [w for w, f in wf_pairs]
    freqs = [f for w, f in wf_pairs]
    chart_data = {
        "category": cate,
        "labels": words,
        "values": freqs}
    #print(chart_data)
    return chart_data, wf_pairs  # chart_data

### Functions for system management
# They can be performed by the system itself periodically.
# The system administrator can also perform them manually.
# This is for the calculation of top words in each category
from collections import Counter
from app_user_keyword_db.models import NewsData
from datetime import datetime, timedelta
from django.db.models import Q, Max, F

allowedNE=['PERSON']
news_categories=['即時', '要聞', '運動', '社會', '全球', '產經', '地方']
# get top words for each category
def calculate_top_person(request):
    
    # Get the latest date in the database
    latest_date = NewsData.objects.aggregate(max_date=Max('date'))['max_date']
    
    # Calculate start date
    start_date = latest_date - timedelta(weeks=4)  # 4 weeks ago
    
    top_cate_ner_words={}
    words_all=[]
    for category in news_categories:
        
        # Use Django's ORM to get entities for the given category
        entities_list = list(NewsData.objects.filter(category=category).filter(date__gte=start_date, date__lte=latest_date).values_list('entities', flat=True))
                
        # Process the retrieved entities
        words_group = []
        for entities in entities_list:
            if entities:  # Check if entities is not None
                words_group += eval(entities)

        # concatenate all terms
        words_all += words_group

        # Get top words by calling ne_word_frequency() function
        topwords = ne_word_frequency( words_group )
        top_cate_ner_words[category] = topwords

    topwords_all = ne_word_frequency(words_all)
    top_cate_ner_words['全部'] = topwords_all
    
    # save it to db
    # save it to db
    for category, top_ners in top_cate_ner_words.items():
        # Convert the list of tuples to string representation for storage
        top_keys_str = str(top_ners)
        
        # Check if an entry for this category already exists
        try:
            # Update existing record
            obj = TopPerson.objects.get(category=category)
            obj.top_keys = top_keys_str
            obj.save()
        except TopPerson.DoesNotExist:
            # Create new record
            TopPerson.objects.create(
                category=category,
                top_keys=top_keys_str
            )
    
    messages.info(request, "Top person calculated and saved successfully")
    # messages.error(request, "Top person calculated and saved successfully")
    return redirect("app_top_person_db:home")  # 你想導回的頁面
    # return render(request, "message.html",msg)  # 渲染訊息頁面
    
    # 只回傳訊息，不渲染網頁
    # Return success message
    # return JsonResponse({"status": "success", "message": "Top person calculated and saved successfully"})



def ne_word_frequency( a_news_ne ):
    filtered_words =[]
    for ner,word in a_news_ne:
        if (len(word) >= 2) & (ner in allowedNE):
            filtered_words.append(word)
    counter = Counter( filtered_words )
    return counter.most_common( 20 )

# NerToken(word='烏克蘭', ner='GPE', idx=(4, 7))  # call function NerToken with three parameters: word, ner, and idx
def NerToken(word, ner, idx):
    # print(ner,word)
    return ner,word

print("app_news_analysis--類別熱門人物db載入成功!")




app_news_analysis--類別熱門人物db載入成功!
